# 🎯 Architecture: Data Pipeline Patterns

**Advanced Big Data Concepts for Interview Preparation**

- **Created:** 2026-02-28
- **Focus:** Lambda Architecture, Kappa Architecture, Batch vs Streaming
- **Tags:** `architecture` | `lambda` | `streaming` | `citi-prep`

## 📖 The Core Concept in Plain Language

Historically, data was processed entirely in **Batch** (e.g., a nightly 2 AM job that processes all of yesterday's files). Business users, however, want their dashboards to reflect reality *right now*. This birthed **Streaming** (processing data milliseconds after it arrives).

The primary architectural challenge of the last decade has been: How do you support instantaneous stream calculations while still ensuring massive, robust, historical accuracy?

### Why This Matters

- **System Design Interviews:** You will invariably be asked how to design a metrics system (like Twitter trending tags). You must understand these patterns to answer correctly.
- **Trade-offs:** Streaming is fast but computationally expensive and hard to recalculate if buggy. Batch is slow but robust, immutable, and easy to backfill.

### The Key Insight

You don't always have to choose. Modern architectures combine both for optimal results.

## 🔍 The Lambda Architecture

In [ ]:
print("Lambda = Batch + Speed (Streaming) + Serving")
print("\n1. Raw Data enters the system and SPLITS into two identical paths.")
print("2. Path A (Speed Layer): Kafka -> Spark Streaming -> Real-time dashboard view (Fast but volatile).")
print("3. Path B (Batch Layer): S3 -> Nightly Airflow Spark Job -> Aggregated Data Warehouse (Slow but immutable/accurate).")
print("4. Serving Layer: The dashboard queries both, combining the robust Batch foundation with the last 2 hours of Speed metrics.")

## 🏗️ The Problem with Lambda (Code Duplication)

Lambda architecture forces you to maintain two separate codebases that do exactly the same math. 

You write a Java/Flink job to do the math on the streaming packets. You write a PySpark/SQL job to do the exact same math on the 5TB nightly batch file. 

If business logic changes, you must perfectly synchronize updates to both codebases, or your real-time dashboard will disagree with your historical dashboard.

In [ ]:
print("Lambda Disadvantages:")
print("- Double the maintenance.")
print("- Double the testing.")
print("- Complex serving layer logic to stitch real-time with history without double-counting.")

## 🔄 The Kappa Architecture

Kappa Architecture proposes a radical shift: **Everything is a Stream.** 

There is no Batch layer. You use a massive append-only log (like Apache Kafka) configured to retain data forever (or for years).

In [ ]:
print("Kappa Flow:")
print("Raw Data -> Apache Kafka -> Stream Processing (Flink/Spark) -> Database")

print("\nHow do you handle 'Batch' concepts like backfilling data or re-running buggy logic?")
print("You simply spin up a new streaming job, tell it to reset its Kafka consumer offset to 'Year 2021', and let it rip through the immutable log from the beginning at high speed until it catches up to the present.")

## 🌍 Real-World Example: Delta Lake (The Hybrid Compromise)

**The Citi Context:** Maintaining 3 years of Peta-scale data in Kafka is impossibly expensive. But managing dual Lambda architectures is impossibly complex for our engineering team sizing. 

We adopted **Data Lakehouse** patterns (Delta/Iceberg).

### The Delta Format
Delta Lake sits on top of cheap S3 object storage but adds ACID transactions. Data streams land in S3. Batch jobs process it all in one unified API.

In [ ]:
# The magic of Spark Structured Streaming + Delta format
print("# One codebase handles both batch and streaming.")
print("# Batch Execution:")
print("df = spark.read.format('delta').load('/path/to/table')")
print("\n# Streaming Execution (Reading the exact same files):")
print("df = spark.readStream.format('delta').load('/path/to/table')")

## 🎭 Critical Concept: Idempotency

In [ ]:
print("Idempotency means: No matter how many times you run an operation, the result is identical to running it exactly once.")
print("\nWhy it matters in pipelines: If a Spark job crashes at 99%, and you restart it, it will re-process the data. If your pipeline is not idempotent, it will insert DUPLICATE rows into your target table.")
print("\nHow to achieve it: Use 'Upserts' (MERGE ON key) rather than raw INSERT statements. Or use Overwrite by Partition.")

## 🎤 5 Interview Q&A

### Q1: What is the primary drawback of Lambda Architecture?
**Answer:** Maintenance overhead and code duplication. You must maintain identical algorithmic logic across two entirely different compute paradigms (the Batch cluster and the Streaming engine), which inevitably leads to divergences and difficult debugging.

---

### Q2: How does a Kappa Architecture handle bugs discovered in historical logic?
**Answer:** In Kappa, the entire history of data is stored as immutable events in the streaming broker (e.g., Kafka). When a bug is found, you deploy a patched version of the stream consumer, set it to read from offset 0, spool it through all historical data to write to a new clean database table, and once it catches up to real-time, you switch the UI to point to the new table.

---

### Q3: How do you achieve Exactly-Once processing in messy distributed streams?
**Answer:** You must couple a streaming broker that supports exactly-once semantics (like Kafka's transactional commits or Spark Structured Streaming's checkpointing) with an **Idempotent** sink. Even if the processor accidentally delivers the payload twice during a crash recovery, the database should "Upsert" by Primary Key, ensuring the row count remains perfectly accurate.

---

### Q4: Contrast S3 vs Kafka as an ingestion layer.
**Answer:** S3 is exceptionally cheap, infinitely scalable for petabytes of history, but inherently batch-oriented (polling for file creation incurs massive latency). Kafka is very expensive for petabyte storage, but provides push-based pub/sub routing with millisecond latency. Modern stacks use Kafka for the fast ingestion routing, and drain the data into S3 for the permanent Lakehouse storage.

---

### Q5: What is "Event Time" vs "Processing Time"?
**Answer:** Event time is the timestamp generated by the client's phone when they clicked the button. Processing time is the timestamp generated by the server when it finally received the packet 3 hours later after the phone reconnected to WiFi. Analytics must always group window operations by Event Time, using "Watermark" algorithms to handle late-arriving data.

## 📚 Key Terminology to Master

| Term | Definition | Example |
|------|-----------|----------|
| **Lambda Architecture** | Separate Speed and Batch layers running in parallel. | Kafka+Storm / S3+Hadoop |
| **Kappa Architecture** | Everything is a stream from an immutable log. | Kafka stored endlessly |
| **Idempotency** | $f(f(x)) = f(x)$. Safe to retry blindly without corrupting data. | `UPSERT` / `MERGE` |
| **Watermarking** | Telling a streaming engine how long to wait for late data before closing an aggregation window. | `withWatermark("time", "10 mins")` |
| **Checkpointing** | Saving the exact offset read from a stream so a crashed node can resume perfectly. | Spark RocksDB checkpoints |

## 💼 The Citi Narrative: Avoiding Lambda Duplication

### The Problem
We had a nightly PySpark batch job that processed all trade executions and updated the risk models correctly. However, the trading desks wanted the dashboard updated intra-day. Standard thinking suggested we bolt-on a Kafka-based Apache Flink cluster to build a Lambda Speed Layer.

### The Solution
Building and managing a separate Flink cluster would double our required headcount. Instead, we transitioned our S3 storage into Delta Lake format to gain ACID properties. We then simply flipped our existing PySpark batch code to use Spark Structured Streaming triggers (`Trigger.AvailableNow`). 

### The Impact
We achieved a unified architecture where a single, unified PySpark codebase could be triggered in "mini-batches" every 5 minutes. We delivered near real-time dashboards (5 min latency was well within SLAs) while completely avoiding the massive engineering overhead of maintaining a dual-path Lambda architecture.

## 💪 Practice Exercises

In [ ]:
# EXERCISE 1: Idempotency Design
# You run a script: UPDATE user SET login_count = login_count + 1 WHERE id=5
# Is this idempotent? Why?
print("No. If the network drops and you retry, the count becomes +2. Non-idempotent.")

In [ ]:
# EXERCISE 2: Lambda serving logic
# How does the UI combine Batch and Speed views seamlessly?
print("UI Query = SELECT sum(x) FROM Nightly_Batch_DB PLUS (SELECT sum(x) FROM Fast_Stream_Redis WHERE time > last_batch_run)")

In [ ]:
# EXERCISE 3: Kappa Scaling Limit
# What is the practical limitation keeping companies from going pure Kappa?
print("Storage costs. Kafka and EventHubs run on highly-performant SSDs attached to compute. Storing 10 years of immutable logs there is mathematically unviable compared to cheap S3 'Cold Storage'.")

## 🎯 Summary: Key Takeaways

### What You've Learned
1. ✅ **Lambda:** Splitting architecture into Batch (accuracy) and Speed (low latency).
2. ✅ **Kappa:** Simplified stream-only architecture constrained mostly by storage economics.
3. ✅ **Idempotency:** The golden rule of Data Engineering. Never write pipelines that break if executed twice.
4. ✅ **Event Time:** Always trust the payload timestamps, never the system processing timestamps.
5. ✅ **Unified Engines:** Spark Structured Streaming and Delta Lake allow developers to blend the lines, writing code once that executes both historically and linearly.

### Interview Confidence Checklist
- [ ] Diagram Lambda architecture on a whiteboard, marking the intersection.
- [ ] Explain how you would recover from a data corruption bug in a Kappa architecture.
- [ ] Demonstrate 'upsert' logic conceptually to achieve idempotency.
- [ ] Articulate the Citi "Choosing Structured Streaming over Flink" story.

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

System Design interviews are rarely about coding. They are about discussing the trade-offs of these exact architectures. 🚀